In [1]:
from pathlib import Path
from collections import defaultdict
import re

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.workbook.properties import CalcProperties

In [2]:
# Change INPUT_DIR if your Excel files are stored elsewhere.
INPUT_DIR = Path('.')
SOURCE_FILES = [
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthBassaResponse.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthLofa.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthMargibi.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthMargibi2.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthMargibiResponse.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthMonsterato.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthNimba.xlsx',
    INPUT_DIR / 'data/redataatindividualpcvlevel/HealthNimba2.xlsx',
]
REFERENCE_FILE = INPUT_DIR / 'data/projectstatusreports/Liberia_PSR_Community_Health_FY26.xlsx'
OUTPUT_FILE = INPUT_DIR / 'Liberia_PSR_Community_Health_FY26_v1.1.xlsx'

QUARTER_STRATEGY = 'latest_nonblank'
for path in [REFERENCE_FILE, *SOURCE_FILES]:
    if not path.is_file():
        raise FileNotFoundError(f'Required Excel file missing: {path.resolve()}')
if OUTPUT_FILE.resolve() in {p.resolve() for p in [REFERENCE_FILE, *SOURCE_FILES]}:
    raise ValueError('The output must not overwrite a source or the Health PSR reference.')

print('HEALTH-ONLY reference:', REFERENCE_FILE.resolve())
print('Volunteer source files:', *(p.name for p in SOURCE_FILES), sep='\n  ')

HEALTH-ONLY reference: /Users/ajgrover/Documents/Scripting/liberia-annual-reporting-26/data/projectstatusreports/Liberia_PSR_Community_Health_FY26.xlsx
Volunteer source files:
  HealthBassaResponse.xlsx
  HealthLofa.xlsx
  HealthMargibi.xlsx
  HealthMargibi2.xlsx
  HealthMargibiResponse.xlsx
  HealthMonsterato.xlsx
  HealthNimba.xlsx
  HealthNimba2.xlsx


In [3]:
def numeric(value):
    """Extract an Excel number; preserve missing versus actual zero."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return value
    if isinstance(value, str):
        value = value.strip().replace(',', '')
        if value:
            try:
                return float(value) if any(x in value.lower() for x in ('.', 'e')) else int(value)
            except ValueError:
                pass
    return None


def label(value):
    return str(value).strip() if value is not None else ''


def latest_quarter(worksheet, excel_row, indices):
    """Select the last numeric quarterly cell, not the sum across periods."""
    if QUARTER_STRATEGY != 'latest_nonblank':
        raise ValueError(f'Unsupported quarter strategy: {QUARTER_STRATEGY}')
    values = [(q, numeric(worksheet.cell(excel_row, col).value))
              for q, col in sorted(indices.items())]
    values = [(q, value) for q, value in values if value is not None]
    if not values:
        return None, ''
    q, value = values[-1]
    if value < 0:
        raise ValueError(f'Negative summary in {worksheet.title} row {excel_row} (Q{q})')
    return value, f'Q{q}'


def health_psr_blocks(ws):
    """Read all Health PSR indicator blocks and their precise disaggregations."""
    blocks, current = [], None
    category = None
    objective = None
    for row in range(4, ws.max_row + 1):
        if label(ws.cell(row, 1).value):
            objective = label(ws.cell(row, 1).value)
        cat_value = label(ws.cell(row, 2).value)
        if cat_value in ('Output', 'Outcome'):
            category = cat_value
        indicator_description = label(ws.cell(row, 3).value)
        match = re.match(r'^(HE_[A-Z0-9_]+)\s*:', indicator_description)
        if match:
            if current is not None:
                raise ValueError(f'Missing Total row before {match.group(1)}; prior indicator: {current["code"]}')
            if category not in ('Output', 'Outcome'):
                raise ValueError(f'Missing category for {match.group(1)} at row {row}')
            current = dict(code=match.group(1), category=category,
                           description=indicator_description, objective=objective,
                           first=row, rows={})
        if current is None:
            continue
        dim1 = label(ws.cell(row, 4).value)
        dim2 = label(ws.cell(row, 5).value)
        if dim1 == 'Total':
            current['last'] = row - 1
            current['total'] = row
            current['reference_reached'] = numeric(ws.cell(row, 7).value)
            current['reference_achieved'] = numeric(ws.cell(row, 8).value)
            current['fy_target'] = ws.cell(row, 10).value
            current['reporter_target'] = ws.cell(row, 13).value
            blocks.append(current)
            current = None
        elif dim1 or dim2:
            key = (dim1, dim2)
            if key in current['rows']:
                raise ValueError(f'Duplicate disaggregation in Health PSR: {current["code"]}, {key}')
            current['rows'][key] = row
    if current is not None:
        raise ValueError(f'No Total row for final Health PSR indicator {current["code"]}')
    if not blocks:
        raise ValueError('No HE_* indicators found in the supplied Health PSR')
    if len({b['code'] for b in blocks}) != len(blocks):
        raise ValueError('Duplicate indicator code in Health PSR')
    return blocks


def read_health_source(path, indicator_map):
    """Extract PSR-compatible source rows and flags for positive activity-only data."""
    workbook = load_workbook(path, data_only=True, read_only=False)
    details, reporters, numeric_reporters = [], defaultdict(set), defaultdict(set)
    excluded, flags = [], []
    try:
        for ws in workbook.worksheets:
            code = ws.title.strip()
            if code not in indicator_map:
                excluded.append((path.name, code, 'Indicator absent from supplied Community Health PSR'))
                continue
            source_description = label(ws['B4'].value)
            if not source_description.startswith(code + ':'):
                raise ValueError(f'Source indicator mismatch: {path.name} / {code}: {source_description}')
            source_kind = 'Outcome' if 'Outcome Indicator' in label(ws['A4'].value) else 'Output'
            if source_kind != indicator_map[code]['category']:
                raise ValueError(f'Category mismatch for {path.name}/{code}: {source_kind} vs Health PSR {indicator_map[code]["category"]}')

            volunteer = label(ws['B3'].value).split(' (')[0].strip()
            if not volunteer:
                raise ValueError(f'Missing volunteer identity: {path.name}/{code}')
            reporters[code].add(volunteer)

            qcols = {'Reached': {}, 'Achieved': {}}
            verification_col, activities = None, {'Reached': [], 'Achieved': []}
            for col in range(1, ws.max_column + 1):
                header = label(ws.cell(7, col).value)
                match = re.fullmatch(r'(Reached|Achieved)\s+Q([1-4])', header, flags=re.I)
                if match:
                    qcols[match.group(1).capitalize()][int(match.group(2))] = col
                elif 'Latest Q Staff Verified' in header:
                    verification_col = col
            if not qcols['Reached'] or (source_kind == 'Outcome' and not qcols['Achieved']):
                raise ValueError(f'Missing quarterly headers in {path.name}/{code}')
            if verification_col is None:
                raise ValueError(f'Missing verification column in {path.name}/{code}')
            for col in range(verification_col + 1, ws.max_column + 1):
                header = label(ws.cell(7, col).value).capitalize()
                if header in activities:
                    activities[header].append(col)

            current_sex = ''
            for row in range(8, min(ws.max_row, 1000) + 1):
                first = label(ws.cell(row, 1).value)
                second = label(ws.cell(row, 2).value)
                if first.lower() == 'key:':
                    break
                if first.startswith('Sex:'):
                    current_sex = first
                if first.startswith(('Other:', 'Intervention type:')):
                    group = (first, '')
                elif current_sex and second.startswith(('Age:', 'Individual:')):
                    group = (current_sex, second)
                elif first.startswith('Sex:') and not second:
                    group = (first, '')
                else:
                    continue

                reached, rq = latest_quarter(ws, row, qcols['Reached'])
                achieved, aq = latest_quarter(ws, row, qcols['Achieved'])
                raw_r = {q: numeric(ws.cell(row, col).value) for q, col in qcols['Reached'].items()}
                raw_a = {q: numeric(ws.cell(row, col).value) for q, col in qcols['Achieved'].items()}
                if reached is not None or achieved is not None:
                    numeric_reporters[code].add(volunteer)
                status = label(ws.cell(row, verification_col).value)
                details.append(dict(
                    Source=path.name, Volunteer=volunteer, Indicator=code,
                    Dimension1=group[0], Dimension2=group[1],
                    Reached=reached, Achieved=achieved, ReachedQuarter=rq, AchievedQuarter=aq,
                    Verification=status, Sheet=ws.title, ExcelRow=row,
                    RawReached=raw_r, RawAchieved=raw_a,
                ))

                for measure, quarterly_value in (('Reached', reached), ('Achieved', achieved)):
                    if quarterly_value is not None:
                        continue
                    positive_activity_cells = []
                    for col in activities[measure]:
                        activity_value = numeric(ws.cell(row, col).value)
                        if activity_value is not None and activity_value > 0:
                            positive_activity_cells.append(f'{get_column_letter(col)}{row}={activity_value:g}')
                    if positive_activity_cells:
                        flags.append(dict(
                            Source=path.name, Volunteer=volunteer, Indicator=code,
                            Dimension1=group[0], Dimension2=group[1], Measure=measure,
                            ActivityCells=', '.join(positive_activity_cells),
                            Verification=status, Sheet=ws.title, ExcelRow=row,
                            Action='Quarterly summary missing; verify manually; activity numbers NOT added to PSR',
                        ))
    finally:
        workbook.close()
    return details, reporters, numeric_reporters, excluded, flags

In [4]:
# Everything on the official report is derived from this Health reference workbook.
report_wb = load_workbook(REFERENCE_FILE)
report_ws = report_wb.worksheets[0]
if 'Community Health' not in label(report_ws['A1'].value):
    raise ValueError('The selected PSR reference does not appear to be the Community Health report.')
blocks = health_psr_blocks(report_ws)
block_map = {b['code']: b for b in blocks}

# Snapshot authoritative data before writing to make accidental overwrites detectable.
PROTECTED_COLS = (1, 2, 3, 4, 5, 6, 10, 13)  # A-F, J, M
original_protected = {
    (row, col): report_ws.cell(row, col).value
    for row in range(1, report_ws.max_row + 1)
    for col in PROTECTED_COLS
}
original_title_and_assignment = (report_ws['A1'].value, report_ws['A2'].value)

all_details = []
reporters = defaultdict(set)
numeric_reporters = defaultdict(set)
excluded_sheets = []
activity_only_flags = []
source_volunteer_indicator = {}

for path in SOURCE_FILES:
    details, found, numeric_found, excluded, flags = read_health_source(path, block_map)
    for code, volunteer_names in found.items():
        for volunteer in volunteer_names:
            unique_id = (code, volunteer.casefold())
            if unique_id in source_volunteer_indicator:
                raise ValueError(
                    f'Duplicate Health indicator report for {volunteer} ({code}): '
                    f'{source_volunteer_indicator[unique_id]} and {path.name}. '
                    'Review manually instead of double-counting.'
                )
            source_volunteer_indicator[unique_id] = path.name
        reporters[code].update(volunteer_names)
    for code, volunteer_names in numeric_found.items():
        numeric_reporters[code].update(volunteer_names)
    all_details.extend(details)
    excluded_sheets.extend(excluded)
    activity_only_flags.extend(flags)

if not all_details:
    raise ValueError('None of the eight volunteer workbooks contained Community Health PSR indicators.')

source_volunteers = sorted({(d['Source'], d['Volunteer']) for d in all_details})
print('Health indicators in PSR:', len(blocks))
print('Health source disaggregation rows:', len(all_details))
print('Distinct volunteers with included Health indicators:', len(source_volunteers))
print('Excluded non-PSR sheets:', len(excluded_sheets))
print('Positive activity-only measure flags:', len(activity_only_flags))

Health indicators in PSR: 19
Health source disaggregation rows: 859
Distinct volunteers with included Health indicators: 7
Excluded non-PSR sheets: 16
Positive activity-only measure flags: 20


In [5]:
# Aggregate by (indicator code, exact PSR column D, exact PSR column E).
agg = defaultdict(lambda: {
    'Reached': 0, 'Achieved': 0, 'has_reached': False, 'has_achieved': False
})
for record in all_details:
    key = (record['Indicator'], record['Dimension1'], record['Dimension2'])
    if key[1:] not in block_map[key[0]]['rows']:
        if record['Reached'] is not None or record['Achieved'] is not None:
            raise ValueError(f'NUMERIC source disaggregation absent from Health PSR: {record["Source"]} / {key}')
        # A blank source-only disaggregation is not a count, so it is audit-only.
        continue
    item = agg[key]
    for measure in ('Reached', 'Achieved'):
        value = record[measure]
        if value is not None:
            if value < 0:
                raise ValueError(f'Negative {measure} for {key} in {record["Source"]}')
            item[measure] += value
            item['has_' + measure.lower()] = True

summary = {}
for b in blocks:
    items = [value for (code, _d, _e), value in agg.items() if code == b['code']]
    has_reached = any(v['has_reached'] for v in items)
    has_achieved = any(v['has_achieved'] for v in items)
    summary[b['code']] = {
        'reached': sum(v['Reached'] for v in items) if has_reached else None,
        'achieved': sum(v['Achieved'] for v in items) if has_achieved else None,
        'pcvs_with_sheet': len(reporters[b['code']]),
        'pcvs_with_numeric': len(numeric_reporters[b['code']]),
    }
    print(f"{b['code']:<22} | {b['category']:<7} | "
          f"PCVs={summary[b['code']]['pcvs_with_sheet']:<2} | "
          f"Reached={str(summary[b['code']]['reached']):<6} | "
          f"Achieved={summary[b['code']]['achieved']}")

HE_MNCH_250_A          | Output  | PCVs=4  | Reached=79     | Achieved=None
HE_MNCH_999_PDI_112    | Output  | PCVs=3  | Reached=20     | Achieved=None
HE_MNCH_029_A          | Outcome | PCVs=4  | Reached=70     | Achieved=69
HE_MNCH_999_PDI_113    | Output  | PCVs=2  | Reached=12     | Achieved=None
HE_MNCH_999_PDI_64     | Output  | PCVs=6  | Reached=60     | Achieved=None
HE_MNCH_999_PDI_65     | Outcome | PCVs=7  | Reached=68     | Achieved=66
HE_MNCH_079            | Output  | PCVs=3  | Reached=112    | Achieved=None
HE_MNCH_083            | Outcome | PCVs=3  | Reached=82     | Achieved=64
HE_MNCH_085            | Outcome | PCVs=3  | Reached=78     | Achieved=78
HE_MNCH_090            | Outcome | PCVs=3  | Reached=88     | Achieved=83
HE_MNCH_093            | Outcome | PCVs=3  | Reached=81     | Achieved=80
HE_MNCH_134            | Outcome | PCVs=3  | Reached=42     | Achieved=33
HE_MNCH_242            | Outcome | PCVs=3  | Reached=79     | Achieved=79
HE_MNCH_114            | Out

In [6]:
# Write into a COPY in memory of the original Health PSR.
# Only G, H, I, K, L are changed (all source objectives/targets remain untouched).
for b in blocks:
    code = b['code']
    first, last, total = b['first'], b['last'], b['total']
    for row in range(first, total + 1):
        for col in (7, 8, 9, 11, 12):
            report_ws.cell(row, col).value = None

    for (d1, d2), row in b['rows'].items():
        measures = agg.get((code, d1, d2))
        if measures is None:
            continue
        if measures['has_reached']:
            report_ws.cell(row, 7).value = measures['Reached']
        if b['category'] == 'Outcome' and measures['has_achieved']:
            report_ws.cell(row, 8).value = measures['Achieved']
        if b['category'] == 'Outcome':
            report_ws.cell(row, 9).value = (
                f'=IF(OR(G{row}="",H{row}=""),"",IFERROR(H{row}/G{row},""))'
            )
            report_ws.cell(row, 9).number_format = '0.0%'

    report_ws.cell(total, 7).value = (
        f'=IF(COUNT(G{first}:G{last})=0,"",SUM(G{first}:G{last}))'
    )
    report_ws.cell(total, 12).value = summary[code]['pcvs_with_sheet']
    if b['category'] == 'Outcome':
        report_ws.cell(total, 8).value = (
            f'=IF(COUNT(H{first}:H{last})=0,"",SUM(H{first}:H{last}))'
        )
        report_ws.cell(total, 9).value = (
            f'=IF(OR(G{total}="",H{total}=""),"",IFERROR(H{total}/G{total},""))'
        )
        numerator = f'H{total}'
    else:
        report_ws.cell(total, 8).value = 'N/A'
        report_ws.cell(total, 9).value = 'N/A'
        numerator = f'G{total}'
    # Only J is used for targets; zero/empty target results in blank %, not infinity.
    report_ws.cell(total, 11).value = (
        f'=IF(OR(J{total}="",J{total}=0,{numerator}=""),"",'
        f'IFERROR({numerator}/J{total},""))'
    )
    report_ws.cell(total, 7).number_format = '#,##0'
    report_ws.cell(total, 9).number_format = '0.0%' if b['category'] == 'Outcome' else 'General'
    report_ws.cell(total, 11).number_format = '0.0%'

# Assert that every authoritative template field survived byte-for-byte as a cell value.
changed_protected = [
    (row, col) for (row, col), original in original_protected.items()
    if report_ws.cell(row, col).value != original
]
if changed_protected:
    raise AssertionError(f'Health PSR objective/target fields were modified: {changed_protected[:8]}')
assert (report_ws['A1'].value, report_ws['A2'].value) == original_title_and_assignment
report_wb.calculation = CalcProperties(calcMode='auto', fullCalcOnLoad=True, forceFullCalc=True)
print('PSR template preserved; aggregation fields updated in memory.')

PSR template preserved; aggregation fields updated in memory.


In [7]:
# Auxiliary worksheets: readable headers, frozen panes and provenance columns.
def style_auxiliary_sheet(ws, widths, last_col, freeze='F5'):
    for row_number, fill, size in [(1, '123B4A', 14), (4, '175A6D', 10)]:
        for col in range(1, last_col + 1):
            cell = ws.cell(row_number, col)
            cell.fill = PatternFill('solid', fgColor=fill)
            cell.font = Font(bold=True, color='FFFFFF', size=size)
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        ws.row_dimensions[row_number].height = 34 if row_number == 1 else 42
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    ws.freeze_panes = freeze
    ws.sheet_view.showGridLines = False


def write_table(ws, columns, rows):
    for col, name in enumerate(columns, start=1):
        ws.cell(4, col).value = name
    for idx, item in enumerate(rows, start=5):
        for col, value in enumerate(item, start=1):
            if value is not None:
                ws.cell(idx, col).value = value
    if rows:
        ws.auto_filter.ref = f'A4:{get_column_letter(len(columns))}{4 + len(rows)}'


audit = report_wb.create_sheet('Source Breakdown')
audit['A1'] = 'HEALTH FY2026  |  SOURCE BREAKDOWN'
audit['A2'] = ('Eight files; exact Health PSR indicator/sex/disaggregation; '
               'latest numeric quarterly summary per measure. Activity-level values NOT summed.')
audit_cols = [
    'Source workbook', 'Volunteer', 'Indicator', 'Disaggregation D', 'Disaggregation E',
    'Reached (selected)', 'Achieved (selected)', 'Reached quarter', 'Achieved quarter',
    'Latest Q staff verification', 'Source sheet', 'Excel row',
    'Reached Q1', 'Reached Q2', 'Reached Q3', 'Reached Q4',
    'Achieved Q1', 'Achieved Q2', 'Achieved Q3', 'Achieved Q4',
]
audit_rows = []
for d in all_details:
    audit_rows.append([
        d['Source'], d['Volunteer'], d['Indicator'], d['Dimension1'], d['Dimension2'],
        d['Reached'], d['Achieved'], d['ReachedQuarter'], d['AchievedQuarter'],
        d['Verification'], d['Sheet'], d['ExcelRow'],
        *(d['RawReached'].get(q) for q in range(1, 5)),
        *(d['RawAchieved'].get(q) for q in range(1, 5)),
    ])
write_table(audit, audit_cols, audit_rows)
style_auxiliary_sheet(audit, {
    'A': 26, 'B': 30, 'C': 22, 'D': 19, 'E': 51,
    'F': 19, 'G': 19, 'H': 17, 'I': 17, 'J': 26,
    'K': 22, 'L': 13, **{get_column_letter(i): 15 for i in range(13, 21)}
}, 20)
for row in range(5, 5 + len(audit_rows)):
    for col in (6, 7, 12, 13, 14, 15, 16, 17, 18, 19, 20):
        audit.cell(row, col).number_format = '#,##0'

In [8]:
comparison = report_wb.create_sheet('Reference Comparison')
comparison['A1'] = 'HEALTH FY2026  |  AGGREGATED vs ORIGINAL PSR'
comparison['A2'] = ('Reference totals may cover more volunteers than the eight uploaded files. '
                    'Differences are intentionally left visible, not reconciled by assumption.')
comparison_columns = [
    'Health indicator', 'Category', 'PCVs with indicator sheet', 'PCVs with quarterly counts',
    'Aggregated reached', 'Original PSR reached', 'Reached difference',
    'Aggregated achieved', 'Original PSR achieved', 'Achieved difference',
    'Health FY target', 'Target reporting PCVs', 'Health PSR indicator definition',
]
comparison_rows = []
for b in blocks:
    info = summary[b['code']]
    r, a = info['reached'], info['achieved']
    ref_r, ref_a = b['reference_reached'], b['reference_achieved']
    comparison_rows.append([
        b['code'], b['category'], info['pcvs_with_sheet'], info['pcvs_with_numeric'],
        r, ref_r, r-ref_r if r is not None and ref_r is not None else None,
        a if b['category'] == 'Outcome' else None,
        ref_a if b['category'] == 'Outcome' else None,
        a-ref_a if b['category'] == 'Outcome' and a is not None and ref_a is not None else None,
        b['fy_target'], b['reporter_target'], b['description'],
    ])
write_table(comparison, comparison_columns, comparison_rows)
style_auxiliary_sheet(comparison, {
    'A': 23, 'B': 13, 'C': 22, 'D': 24, 'E': 19, 'F': 21, 'G': 19,
    'H': 20, 'I': 21, 'J': 20, 'K': 20, 'L': 22, 'M': 70,
}, 13, freeze='E5')
for row in range(5, 5 + len(comparison_rows)):
    for col in range(3, 13):
        comparison.cell(row, col).number_format = '#,##0'
    for col in (7, 10):
        if isinstance(comparison.cell(row, col).value, (int, float)) and comparison.cell(row, col).value != 0:
            comparison.cell(row, col).font = Font(bold=True, color='9C3428')

In [9]:
review = report_wb.create_sheet('Activity-Only Review')
review['A1'] = 'HEALTH FY2026 | POSITIVE ACTIVITY-ONLY COUNTS'
review['A2'] = ('These source activity cells have no numeric quarterly summary for the same measure. '
                'They have NOT been added to the official aggregated PSR; review the original source worksheets.')
flag_cols = [
    'Source workbook', 'Volunteer', 'Indicator', 'Disaggregation D', 'Disaggregation E',
    'Missing summary metric', 'Positive source activity cells (not summed)',
    'Latest Q staff verification', 'Source sheet', 'Excel row', 'Follow-up needed',
]
flag_rows = [
    [f['Source'], f['Volunteer'], f['Indicator'], f['Dimension1'], f['Dimension2'],
     f['Measure'], f['ActivityCells'], f['Verification'], f['Sheet'], f['ExcelRow'], f['Action']]
    for f in activity_only_flags
]
write_table(review, flag_cols, flag_rows)
style_auxiliary_sheet(review, {
    'A': 26, 'B': 30, 'C': 22, 'D': 20, 'E': 50, 'F': 20,
    'G': 58, 'H': 26, 'I': 22, 'J': 13, 'K': 85,
}, 11, freeze='G5')

excluded = report_wb.create_sheet('Excluded Sheets')
excluded['A1'] = 'HEALTH FY2026 | EXCLUDED NON-PSR INDICATORS'
excluded['A2'] = ('Source sheet codes not defined by the supplied Community Health PSR. '
                  'This excludes generic, malaria and any other unrelated/non-template indicators.')
write_table(excluded, ['Source workbook', 'Excluded sheet / indicator', 'Reason'], excluded_sheets)
style_auxiliary_sheet(excluded, {'A': 27, 'B': 30, 'C': 76}, 3, freeze='B5')

In [10]:
notes = report_wb.create_sheet('Aggregation Notes')
notes['A1'] = 'FY2026 COMMUNITY HEALTH AGGREGATION | METHOD & SCOPE'
notes['A2'] = 'Objectives, indicators, targets and layout come ONLY from Liberia_PSR_Community_Health_FY26.xlsx.'
notes_data = [
    ('Reference PSR', REFERENCE_FILE.name),
    ('Source files (8)', ', '.join(p.name for p in SOURCE_FILES)),
    ('Original PSR project title', label(original_title_and_assignment[0])),
    ('Original assigned-volunteer statement', label(original_title_and_assignment[1])),
    ('Indicators in the Health PSR', len(blocks)),
    ('Distinct source volunteers with PSR indicators', len(source_volunteers)),
    ('Extracted Health disaggregation rows', len(all_details)),
    ('Excluded source sheets', len(excluded_sheets)),
    ('Positive activity-only measure flags', len(activity_only_flags)),
    ('Source summarization', 'Latest available numeric Reached/Achieved Q1-Q4 PER row; never add quarters/activities'),
    ('Disaggregation mapping', 'Indicator code plus exact Health PSR D/E labels (sex + age, occupation, trimester or Other)'),
    ('Quarterly empty cells', 'Blank is missing, never treated as zero; source explicit 0 is counted as 0'),
    ('Verification status', 'UNVERIFIED / Qn retained in Source Breakdown. No automatic exclusion of numerical counts'),
    ('Activity-only policy', 'Positive activity counts without a quarterly summary are flagged and excluded from PSR totals'),
    ('Protection of PSR template', 'All A-F, J, M rows preserved, along with title/20-volunteer statement and source formatting'),
    ('Updated PSR columns', 'G: reached, H: achieved, I: % achieved, K: % of FY target, L: reporting PCV count'),
    ('Reporting PCV policy', 'Distinct volunteers with indicator sheet, even when no numeric summary exists; numeric reporters shown separately'),
    ('Output indicator calculation', 'Reached total is sum of matching source disaggregations; achieved is N/A'),
    ('Outcome indicator calculation', 'Reached and achieved are independently aggregated from quarterly summaries'),
    ('FY target percentages', 'Numerator is reached for Output or achieved for Outcome; target comes from Health PSR J'),
    ('Original national totals', 'Comparison-only, NOT reused, adjusted, or assumed to be eight-volunteer totals'),
    ('Zero-target indicators', 'Percentage left blank for targets equal to zero; original target remains unchanged'),
    ('Not included', 'Literacy PSR, STEM/Math & Science PSR, GEN indicators, HE_MALARIA indicators, HE_MNCH_128'),
]
for row, (key, value) in enumerate(notes_data, start=4):
    notes.cell(row, 1).value = key
    notes.cell(row, 2).value = value
    notes.cell(row, 1).font = Font(bold=True, color='173A4B')
    if row % 2 == 0:
        notes.cell(row, 1).fill = PatternFill('solid', fgColor='EAF4F5')
        notes.cell(row, 2).fill = PatternFill('solid', fgColor='EAF4F5')
    notes.cell(row, 2).alignment = Alignment(wrap_text=True, vertical='center')
    notes.row_dimensions[row].height = 30 if len(str(value)) > 85 else 22
notes.column_dimensions['A'].width = 43
notes.column_dimensions['B'].width = 118
notes.sheet_view.showGridLines = False
notes.freeze_panes = 'B4'
for col in (1, 2):
    cell = notes.cell(1, col)
    cell.fill = PatternFill('solid', fgColor='123B4A')
    cell.font = Font(bold=True, color='FFFFFF', size=14)
notes.row_dimensions[1].height = 35

# Final invariant check before writing.
assert all(report_ws.cell(row, col).value == original for (row, col), original in original_protected.items())
report_wb.save(OUTPUT_FILE)
report_wb.close()
print('\nSAVED:', OUTPUT_FILE.resolve())
print('Official Health indicators:', len(blocks), '| Source files:', len(SOURCE_FILES),
      '| Volunteers:', len(source_volunteers))
print('Audit tabs: Source Breakdown, Reference Comparison, Activity-Only Review, Excluded Sheets, Aggregation Notes')


SAVED: /Users/ajgrover/Documents/Scripting/liberia-annual-reporting-26/Liberia_PSR_Community_Health_FY26_v1.1.xlsx
Official Health indicators: 19 | Source files: 8 | Volunteers: 7
Audit tabs: Source Breakdown, Reference Comparison, Activity-Only Review, Excluded Sheets, Aggregation Notes


In [11]:
# Compact file-based verification; does not alter the result.
check = load_workbook(OUTPUT_FILE, read_only=False, data_only=False)
try:
    assert len(check.worksheets) == 6
    main = check.worksheets[0]
    assert main['A1'].value == original_title_and_assignment[0]
    assert main['A2'].value == original_title_and_assignment[1]
    for (row, col), val in original_protected.items():
        assert main.cell(row, col).value == val or (val in ('', None) and main.cell(row, col).value in ('', None)), (row, col)
    for b in blocks:
        row = b['total']
        assert main.cell(row, 7).value.startswith('=IF(COUNT(')
        assert main.cell(row, 12).value == summary[b['code']]['pcvs_with_sheet']
        assert main.cell(row, 10).value == b['fy_target']
        assert main.cell(row, 13).value == b['reporter_target']
    print('VALIDATED: all 19 Health indicators, all Health PSR targets, A-F/J/M, formulas and six worksheets.')
finally:
    check.close()

VALIDATED: all 19 Health indicators, all Health PSR targets, A-F/J/M, formulas and six worksheets.
